In [ ]:
# ==========================================
# 1 INSTALL LIBRARIES
# ==========================================

!pip install tensorflow opencv-python scikit-learn gradio -q


# ==========================================
# 2 IMPORT LIBRARIES
# ==========================================

import os
import zipfile
import cv2
import numpy as np
import tensorflow as tf
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


# ==========================================
# 3 EXTRACT ZIP DATASET
# ==========================================

zip_path = "archive (9).zip"
extract_path = "dataset"

with zipfile.ZipFile(zip_path,'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted")


# ==========================================
# 4 DATASET PATH
# ==========================================

dataset_path = "dataset/images/images"

print("Classes Found:", os.listdir(dataset_path))


# ==========================================
# 5 LOAD IMAGES
# ==========================================

IMG_SIZE = 224

X = []
y = []

for class_name in os.listdir(dataset_path):

    class_folder = os.path.join(dataset_path,class_name)

    if os.path.isdir(class_folder):

        for img_name in os.listdir(class_folder):

            img_path = os.path.join(class_folder,img_name)

            image = cv2.imread(img_path)

            if image is None:
                continue

            image = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
            image = image/255.0

            X.append(image)
            y.append(class_name)

X = np.array(X)
y = np.array(y)

print("Total Images:",len(X))


# ==========================================
# 6 LABEL ENCODING
# ==========================================

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("Classes:",encoder.classes_)


# ==========================================
# 7 TRAIN TEST SPLIT
# ==========================================

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:",len(X_train))


# ==========================================
# 8 DEEP LEARNING MODEL
# ==========================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256,activation='relu')(x)
x = Dropout(0.5)(x)

predictions = Dense(len(encoder.classes_),activation='softmax')(x)

model = Model(inputs=base_model.input,outputs=predictions)


# ==========================================
# 9 COMPILE MODEL
# ==========================================

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# 10 TRAIN MODEL
# ==========================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=10,
    batch_size=16
)


# ==========================================
# 11 EVALUATE MODEL
# ==========================================

loss,acc = model.evaluate(X_test,y_test)

print("Model Accuracy:",acc)


# ==========================================
# 12 SAVE MODEL
# ==========================================

model.save("doremon_character_model.keras")

print("Model Saved")




Dataset Extracted
Classes Found: ['Suneo', 'Test', 'Nobita', 'Shizuka', 'Doraemon', 'Gian']
Total Images: 136
Classes: ['Doraemon' 'Gian' 'Nobita' 'Shizuka' 'Suneo' 'Test']
Training Samples: 108
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.2037 - loss: 2.2122 - val_accuracy: 0.1429 - val_loss: 1.8687
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 917ms/step - accuracy: 0.3056 - loss: 1.7056 - val_accuracy: 0.1786 - val_loss: 1.7221
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 723ms/step - accuracy: 0.3889 - loss: 1.4981 - val_accuracy: 0.2143 - val_loss: 1.6158
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.4352 - loss: 1.3906 - val_accuracy: 0.2500 - val_loss: 1.5344
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 712ms/step - accuracy: 0.5370 - loss: 1.2508 - val_accuracy: 0.3214 - val_loss: 1.4593
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 715ms/step - accuracy: 0.6204 - loss: 1.0517 - val_accuracy: 0.3214 - val_loss: 1.4183
Epoch

In [ ]:
# ==========================================
# 13 LOAD MODEL FOR GUI
# ==========================================

model = tf.keras.models.load_model("doremon_character_model.keras")

class_names = encoder.classes_


# ==========================================
# 14 PREDICTION FUNCTION
# ==========================================

def predict_character(image):

    image = cv2.resize(image,(224,224))
    image = image/255.0
    image = np.expand_dims(image,axis=0)

    prediction = model.predict(image)

    class_index = np.argmax(prediction)
    confidence = np.max(prediction)

    result = class_names[class_index]

    return f"Character: {result} | Confidence: {confidence:.2f}"


# ==========================================
# 15 CREATE GUI
# ==========================================

interface = gr.Interface(
    fn=predict_character,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="Doraemon Character Detection AI",
    description="Upload an image to detect Doraemon character"
)


# ==========================================
# 16 RUN GUI
# ==========================================

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7a5aec9951b40751b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
